# E1.2 — Trajectory Extension (Paper 1 extension)

Phase **E1.2** of the extension workflow, parameter-aligned with `Step_2_5b_Extended_Production_100ns.ipynb`.
Under the simplified design, E1.2 and E2 are merged: this notebook runs **one** independent
100 ns trajectory for a single `(system, replica)` pair, from the post-equilibration state
with a deterministic, replica-specific velocity assignment. Run it 27 times — edit
**Run configuration**, execute top to bottom, commit, repeat.

**Starting coordinates.** Sourced from **frame 0 of `{system}_prod.dcd`**, read through the
prmtop so atom order is guaranteed consistent. (The `_equilibrated.pdb` files are mis-ordered
relative to the prmtop — loading one gave a ~1e11 kJ/mol clash; frame 0 of production gives
the correct ~-3.7e4 kJ/mol post-equilibration state.) All three replicates of a system share
this starting structure and differ only in their velocity seed, per the design.

**Parameters carried forward from the original production (Step 2.4/2.5, via Step_2_5b):**
`LangevinIntegrator` (300 K, gamma = 1 ps^-1); MonteCarloBarostat (1 bar); 2 fs; PME at
1.0 nm; HBonds constraints. Force field is encoded in the `.prmtop`. DCD uses the
periodic-system default imaging (wrapped), matching Phase 3. Save interval 1 ps; ~3.4 GB per
trajectory, ~92 GB across the 27.

## 1 — Imports
**In:** `des-peptide` environment (OpenMM 8.5.1, MDTraj). **Out:** namespace; prints versions and platforms.

In [1]:
import os, sys, time, json, hashlib, subprocess
import platform as _platform
from datetime import datetime, timezone
from pathlib import Path

import openmm as mm
import openmm.app as app
import openmm.unit as unit
from openmm import LangevinIntegrator, MonteCarloBarostat, Platform
import mdtraj as md

print(f"OpenMM {mm.__version__} | MDTraj {md.__version__}")
print("Platforms:", [Platform.getPlatform(i).getName() for i in range(Platform.getNumPlatforms())])

OpenMM 8.5.1 | MDTraj 1.11.1
Platforms: ['Reference', 'CPU', 'OpenCL']


## 2 — Fixed protocol constants
Carried forward verbatim from the original production. **Do not edit between runs.**
**Out:** module-level constants.

In [2]:
SYSTEMS = [
    "GGE_water", "CME_water", "YIY_water",
    "GGE_reline", "CME_reline", "YIY_reline",
    "GGE_glyceline", "CME_glyceline", "YIY_glyceline",
]

TEMPERATURE = 300 * unit.kelvin
PRESSURE    = 1 * unit.bar
FRICTION    = 1.0 / unit.picosecond
TIMESTEP    = 2.0 * unit.femtoseconds
PME_CUTOFF  = 1.0 * unit.nanometers

SAVE_INTERVAL   = 500      # 1 ps  (simplified-design spec; Step_2_5b used 1000 / 2 ps)
LOG_INTERVAL    = 5000     # 10 ps (original production value)
STDOUT_INTERVAL = 100000   # 200 ps (Step_2_5b value)

REFERENCE_NS_PER_DAY = 710.0  # E1.1 benchmark (M5 Max, OpenCL); estimates only

## 3 — Run configuration  *(edit this cell per run)*
The only cell that changes between the 27 runs.
**Out:** run-specific parameters.

In [3]:
SYSTEM      = "GGE_water"    # one of SYSTEMS
REPLICA     = 1             # 1, 2, or 3

PROJECT_DIR = Path("~/des-peptide-study").expanduser()
SYSTEMS_DIR = PROJECT_DIR / "systems"
OUTPUT_DIR  = PROJECT_DIR / "extension" / "trajectories"

NS          = 100.0         # production length
PLATFORM    = "OpenCL"      # E1.1 production platform; OpenCL | CUDA | CPU | Reference

## 4 — Deterministic velocity seed
Required by the simplified design (distinct, well-defined velocities per replicate). SHA-256
maps `(system, replica)` to a fixed 31-bit seed. It fixes the integrator noise stream and the
initial velocities, so the starting microstate and replica identity are reproducible — it
does **not** make the GPU trajectory bit-reproducible.
**In:** `SYSTEM`, `REPLICA`. **Out:** `SEED`.

In [4]:
def derive_seed(system: str, replica: int) -> int:
    """Reproducible non-zero 31-bit seed (OpenMM treats 0 as 'choose randomly')."""
    digest = hashlib.sha256(f"{system}_rep{replica}".encode()).digest()
    return int.from_bytes(digest[:4], "big") % (2 ** 31 - 2) + 1

SEED = derive_seed(SYSTEM, REPLICA)
print(f"{SYSTEM} rep{REPLICA} -> seed {SEED}")

GGE_water rep1 -> seed 2002923534


## 5 — Resolve and validate I/O paths
Handles both the per-system subdirectory layout (`systems/{name}/...`, as in Step_2_5b) and a
flat layout. The starting coordinates come from frame 0 of the original production DCD. Fails
fast if inputs are missing.
**In:** `SYSTEM`, `REPLICA`, dirs. **Out:** input/output path variables.

In [5]:
assert SYSTEM in SYSTEMS, f"Unknown system {SYSTEM!r}; expected one of {SYSTEMS}"
if REPLICA not in (1, 2, 3):
    print(f"[warn] replica {REPLICA} is outside the planned set {{1, 2, 3}}", file=sys.stderr)

def resolve(name):
    """First existing path among subdir then flat layouts."""
    for cand in (SYSTEMS_DIR / SYSTEM / name, SYSTEMS_DIR / name):
        if cand.exists():
            return cand
    return None

PRMTOP   = resolve(f"{SYSTEM}.prmtop")
PROD_DCD = resolve(f"{SYSTEM}_prod.dcd")
assert PRMTOP is not None,   f"Topology {SYSTEM}.prmtop not found under {SYSTEMS_DIR}"
assert PROD_DCD is not None, f"Production DCD {SYSTEM}_prod.dcd not found under {SYSTEMS_DIR}"

TAG       = f"{SYSTEM}_rep{REPLICA}"
RUN_DIR   = OUTPUT_DIR / TAG
DCD       = RUN_DIR / f"{TAG}.dcd"
STATE_XML = RUN_DIR / f"{TAG}_final_state.xml"
LOG       = RUN_DIR / f"{TAG}_production.log"
METADATA  = RUN_DIR / f"{TAG}_metadata.json"

print("topology     :", PRMTOP)
print("start coords :", f"{PROD_DCD} (frame 0)")
print("output dir   :", RUN_DIR)

topology     : /Users/rossgibson/des-peptide-study/systems/GGE_water/GGE_water.prmtop
start coords : /Users/rossgibson/des-peptide-study/systems/GGE_water/GGE_water_prod.dcd (frame 0)
output dir   : /Users/rossgibson/des-peptide-study/extension/trajectories/GGE_water_rep1


## 6 — Pre-flight estimate (compute + storage)
Run **before** building to see the cost up front. Starts nothing.
**In:** `NS`, `PRMTOP`. **Out:** `TOTAL_STEPS`, `N_FRAMES`; prints time and DCD-size estimates.

In [6]:
TOTAL_STEPS = int(NS * 1e6 / TIMESTEP.value_in_unit(unit.femtoseconds))
N_FRAMES    = TOTAL_STEPS // SAVE_INTERVAL
EST_HOURS   = NS / REFERENCE_NS_PER_DAY * 24.0

N_ATOMS = app.AmberPrmtopFile(str(PRMTOP)).topology.getNumAtoms()
EST_GB  = N_FRAMES * (N_ATOMS * 12 + 80) / 1e9

print("=" * 64)
print(f"  system / replica : {SYSTEM} / rep{REPLICA}   (seed {SEED})")
print(f"  atoms            : {N_ATOMS}")
print(f"  length           : {NS:g} ns  ({TOTAL_STEPS:,} steps @ {TIMESTEP.value_in_unit(unit.femtoseconds):g} fs)")
print(f"  frames           : {N_FRAMES:,} ({SAVE_INTERVAL} steps / {SAVE_INTERVAL*TIMESTEP.value_in_unit(unit.picoseconds):g} ps each)")
print(f"  platform         : {PLATFORM} (default precision)")
print(f"  est. wall time   : {EST_HOURS:.2f} h  (@ {REFERENCE_NS_PER_DAY:g} ns/day)")
print(f"  est. DCD size    : {EST_GB:.2f} GB")
print("=" * 64)

  system / replica : GGE_water / rep1   (seed 2002923534)
  atoms            : 2838
  length           : 100 ns  (50,000,000 steps @ 2 fs)
  frames           : 100,000 (500 steps / 1 ps each)
  platform         : OpenCL (default precision)
  est. wall time   : 3.38 h  (@ 710 ns/day)
  est. DCD size    : 3.41 GB


## 7 — Build the simulation
Loads the post-equilibration state from frame 0 of `_prod.dcd` (prmtop atom order), builds the
system with the original production parameters, and assigns fresh velocities from `SEED`. No
re-minimisation or re-equilibration; the per-trajectory equilibration cutoff is applied in
E3.1. The starting potential energy is printed as a guard — it must be a sane finite value
(order -3.7e4 kJ/mol), not the ~1e11 a coordinate mismatch produces.
**In:** `PRMTOP`, `PROD_DCD`, `SEED`, `PLATFORM`. **Out:** `simulation`.

In [7]:
def load_start(prmtop_path: Path, prod_dcd_path: Path):
    """Frame 0 of the original production = post-equilibration state, in prmtop atom order."""
    t0 = md.load_frame(str(prod_dcd_path), 0, top=str(prmtop_path))
    return t0.openmm_positions(0), t0.openmm_boxes(0)

RUN_DIR.mkdir(parents=True, exist_ok=True)

prmtop = app.AmberPrmtopFile(str(PRMTOP))
positions, box = load_start(PRMTOP, PROD_DCD)

system = prmtop.createSystem(
    nonbondedMethod=app.PME,
    nonbondedCutoff=PME_CUTOFF,
    constraints=app.HBonds,
)
system.setDefaultPeriodicBoxVectors(*box)

barostat = MonteCarloBarostat(PRESSURE, TEMPERATURE)
barostat.setRandomNumberSeed(SEED)
system.addForce(barostat)

integrator = LangevinIntegrator(TEMPERATURE, FRICTION, TIMESTEP)
integrator.setRandomNumberSeed(SEED)

mm_platform = Platform.getPlatformByName(PLATFORM)
simulation = app.Simulation(prmtop.topology, system, integrator, mm_platform)
simulation.context.setPositions(positions)
simulation.context.setPeriodicBoxVectors(*box)
simulation.context.setVelocitiesToTemperature(TEMPERATURE, SEED)

start_pe = simulation.context.getState(getEnergy=True).getPotentialEnergy()
assert abs(start_pe.value_in_unit(unit.kilojoules_per_mole)) < 1e7, \
    f"Starting PE {start_pe} is non-physical; check the coordinate source."
print(f"built: {system.getNumParticles()} atoms, platform {mm_platform.getName()}, start PE {start_pe}")

built: 2838 atoms, platform OpenCL, start PE -37452.16645410773 kJ/mol


## 8 — Attach reporters
DCD (1 ps, default imaging), StateData to the log file, and a lighter StateData to the
notebook output. Reporter fields mirror the original production. `reporters.clear()` first so
a re-run does not stack duplicates.
**In:** `simulation`, intervals. **Out:** reporters attached.

In [8]:
simulation.reporters.clear()

simulation.reporters.append(
    app.DCDReporter(str(DCD), SAVE_INTERVAL))
simulation.reporters.append(
    app.StateDataReporter(str(LOG), LOG_INTERVAL,
        step=True, time=True, potentialEnergy=True,
        temperature=True, density=True, speed=True))
simulation.reporters.append(
    app.StateDataReporter(sys.stdout, STDOUT_INTERVAL,
        step=True, time=True, temperature=True,
        speed=True, remainingTime=True, totalSteps=TOTAL_STEPS))

print(f"reporters attached; DCD + log every {SAVE_INTERVAL*TIMESTEP.value_in_unit(unit.picoseconds):g} / "
      f"{LOG_INTERVAL*TIMESTEP.value_in_unit(unit.picoseconds):g} ps")

reporters attached; DCD + log every 1 / 10 ps


## 9 — Run production
The long-running cell (~3.4 h per 100 ns at 710 ns/day). Progress prints inline every ~200 ps.
**In:** `simulation`, `TOTAL_STEPS`. **Out:** `WALL_S`, timestamps; DCD and log on disk.

In [9]:
STARTED = datetime.now(timezone.utc)
_t0 = time.time()
simulation.step(TOTAL_STEPS)
WALL_S = time.time() - _t0
FINISHED = datetime.now(timezone.utc)

print(f"ran {TOTAL_STEPS:,} steps in {WALL_S/3600:.2f} h")

#"Step","Time (ps)","Temperature (K)","Speed (ns/day)","Time Remaining"
100000,200.00000000022686,304.8119068420327,0,--
200000,400.00000000118183,304.1026497085167,703,3:24:04
300000,599.9999999996356,305.9722196357399,702,3:23:47
400000,799.9999999949063,300.1233309907068,702,3:23:23
500000,999.9999999901769,300.35102480340544,701,3:23:15
600000,1199.9999999854476,295.2822168362493,701,3:23:03
700000,1399.9999999807183,303.8632881970789,700,3:22:49
800000,1599.9999999759889,292.75471947644326,700,3:22:28
900000,1799.9999999712595,296.3636347353294,700,3:22:08
1000000,1999.9999999665301,301.5977112618597,699,3:21:49
1100000,2199.9999999618008,301.86856125623467,699,3:21:27
1200000,2399.9999999570714,304.2977924807748,699,3:21:09
1300000,2599.999999952342,299.5677564530183,698,3:20:51
1400000,2799.9999999476127,293.5456096335374,698,3:20:35
1500000,2999.9999999428833,302.37006796403887,697,3:20:16
1600000,3199.999999938154,298.4529292939757,697,3:19:56
1700000,3399.9999999334245,293.16

## 10 — Save final state + write provenance
Saves the final State and a metadata JSON. `achieved_ns_per_day` feeds the
post-first-replicate checkpoint and the tau_int re-evaluation.
**In:** run results. **Out:** `STATE_XML`, `METADATA` on disk; commit after this cell.

In [10]:
simulation.saveState(str(STATE_XML))

def git_commit() -> str:
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"],
                                       stderr=subprocess.DEVNULL).decode().strip()
    except Exception:
        return "unknown"

achieved = (TOTAL_STEPS * TIMESTEP.value_in_unit(unit.nanoseconds)) / (WALL_S / 86400.0) if WALL_S else None
metadata = {
    "phase": "E1.2", "system": SYSTEM, "replica": REPLICA, "seed": SEED,
    "ns": NS, "total_steps": TOTAL_STEPS, "save_interval_steps": SAVE_INTERVAL,
    "integrator": "LangevinIntegrator", "temperature_K": 300,
    "friction_per_ps": 1.0, "timestep_fs": 2.0, "barostat": "MonteCarlo (1 bar)",
    "nonbonded": "PME, 1.0 nm cutoff, HBonds constraints",
    "platform": PLATFORM, "precision": "default",
    "topology": str(PRMTOP), "start_coordinates": f"{PROD_DCD} (frame 0)",
    "started_utc": STARTED.isoformat(), "finished_utc": FINISHED.isoformat(),
    "wall_seconds": round(WALL_S, 1),
    "achieved_ns_per_day": round(achieved, 1) if achieved else None,
    "openmm_version": mm.__version__, "mdtraj_version": md.__version__,
    "python_version": _platform.python_version(),
    "host": _platform.node(), "git_commit": git_commit(),
}
METADATA.write_text(json.dumps(metadata, indent=2))
print(f"[done] {TAG}: {WALL_S/3600:.2f} h, {metadata['achieved_ns_per_day']} ns/day -> {RUN_DIR}")
metadata

[done] GGE_water_rep1: 3.94 h, 609.2 ns/day -> /Users/rossgibson/des-peptide-study/extension/trajectories/GGE_water_rep1


{'phase': 'E1.2',
 'system': 'GGE_water',
 'replica': 1,
 'seed': 2002923534,
 'ns': 100.0,
 'total_steps': 50000000,
 'save_interval_steps': 500,
 'integrator': 'LangevinIntegrator',
 'temperature_K': 300,
 'friction_per_ps': 1.0,
 'timestep_fs': 2.0,
 'barostat': 'MonteCarlo (1 bar)',
 'nonbonded': 'PME, 1.0 nm cutoff, HBonds constraints',
 'platform': 'OpenCL',
 'precision': 'default',
 'topology': '/Users/rossgibson/des-peptide-study/systems/GGE_water/GGE_water.prmtop',
 'start_coordinates': '/Users/rossgibson/des-peptide-study/systems/GGE_water/GGE_water_prod.dcd (frame 0)',
 'started_utc': '2026-06-05T07:37:24.530434+00:00',
 'finished_utc': '2026-06-05T11:33:46.367680+00:00',
 'wall_seconds': 14181.8,
 'achieved_ns_per_day': 609.2,
 'openmm_version': '8.5.1',
 'mdtraj_version': '1.11.1',
 'python_version': '3.11.15',
 'host': 'Rosss-MacBook-Pro.local',
 'git_commit': '87c2f99'}

In [11]:
from mdtraj.formats import DCDTrajectoryFile
import mdtraj as md, numpy as np, json

with DCDTrajectoryFile(str(DCD)) as f:
    n = len(f)
expected = TOTAL_STEPS // SAVE_INTERVAL
print(f"frames    : {n:,} / expected {expected:,}  {'OK' if n == expected else 'MISMATCH'}")

last = md.load_frame(str(DCD), n - 1, top=str(PRMTOP))
print(f"last frame: NaN={bool(np.isnan(last.xyz).any())}  box(nm)={last.unitcell_lengths[0].round(3)}")
print(f"files     : {sorted(p.name for p in RUN_DIR.iterdir())}")
print(f"metadata  : steps={json.loads(METADATA.read_text())['total_steps']:,}, "
      f"{json.loads(METADATA.read_text())['achieved_ns_per_day']} ns/day")

frames    : 100,000 / expected 100,000  OK
last frame: NaN=False  box(nm)=[3.045 3.045 3.045]
files     : ['GGE_water_rep1.dcd', 'GGE_water_rep1_final_state.xml', 'GGE_water_rep1_metadata.json', 'GGE_water_rep1_production.log']
metadata  : steps=50,000,000, 609.2 ns/day
